# Transformar Datos Locations
- Eliminar registros null en el campo location_id
- Eliminar registros duplicados(identicos)
- Eliminar registros duplicados en base a un campo
- Convertir la columna al tipo de dato correcto 
- Escribir los datos transformados al schema silver


## 01. Eliminar registros null en el campo location_id

In [0]:
SELECT *FROM zenviro.bronze.v_locations;

In [0]:
SELECT *FROM zenviro.bronze.v_locations Where location_id IS NOT NULL;

## 02. Eliminar registros duplicados(identicos)

In [0]:
SELECT *
FROM zenviro.bronze.v_locations 
Where location_id IS NOT NULL
ORDER BY location_id;

In [0]:
SELECT 
DISTINCT(*) 
FROM zenviro.bronze.v_locations 
Where location_id IS NOT NULL
ORDER BY location_id;

## 03. Eliminar registros duplicados en base a un campo

In [0]:
-- PRIMER FORMA: ESTA NO ES LA MANERA CORRECTA
SELECT 
    location_id,
    MAX(created_timestamp),
    MAX(location_name),
    MAX(address),
    MAX(city),
    MAX(country)
FROM zenviro.bronze.v_locations 
Where location_id IS NOT NULL
GROUP BY location_id
ORDER BY location_id;

In [0]:
CREATE OR REPLACE TEMPORARY VIEW tv_location_distinct
AS
SELECT 
DISTINCT(*) 
FROM zenviro.bronze.v_locations 
Where location_id IS NOT NULL
ORDER BY location_id;

In [0]:
--SEGUNDA FORMA: UTILIZADO O EXPRESIONES  DE TABLA
SELECT 
    location_id,
    MAX(created_timestamp) as max_created_timestamp
FROM tv_location_distinct
GROUP BY location_id;

In [0]:
WITH cte_max
AS
(
  SELECT 
    location_id,
    MAX(created_timestamp) as max_created_timestamp
    FROM tv_location_distinct
    GROUP BY location_id
)
SELECT 
    CAST(t.created_timestamp AS TIMESTAMP) AS created_timestamp,
    t.location_id,
    t.location_name,
    t.address,
    t.city,
    t.country
   
FROM tv_location_distinct  t
INNER JOIN cte_max c
ON t.location_id = c.location_id AND t.created_timestamp =c.max_created_timestamp


## 04. Convertir la columna al tipo de dato correcto

In [0]:
WITH cte_max
AS
(
  SELECT 
    location_id,
    MAX(created_timestamp) as max_created_timestamp
    FROM tv_location_distinct
    GROUP BY location_id
)
SELECT 
    CAST(t.created_timestamp AS TIMESTAMP) AS created_timestamp,
    t.location_id,
    t.location_name,
    t.address,
    t.city,
    t.country
   
FROM tv_location_distinct  t
INNER JOIN cte_max c
ON t.location_id = c.location_id AND t.created_timestamp =c.max_created_timestamp

## 05. Escribir los datos transformados al schema silver

In [0]:
    CREATE TABLE IF NOT EXISTS zenviro.silver.locations
    AS
            WITH cte_max
            AS
            (
        SELECT 
            location_id,
            MAX(created_timestamp) as max_created_timestamp
            FROM tv_location_distinct
            GROUP BY location_id
            )
        SELECT 
            CAST(t.created_timestamp AS TIMESTAMP) AS created_timestamp,
            t.location_id,
            t.location_name,
            t.address,
            t.city,
            t.country
        
        FROM tv_location_distinct  t
        INNER JOIN cte_max c
        ON t.location_id = c.location_id AND t.created_timestamp =c.max_created_timestamp


In [0]:
SELECT *FROM zenviro.silver.locations

In [0]:
DESCRIBE EXTENDED zenviro.silver.locations